# FS11-12 stable generation


In [ ]:
import os, json, math, random, time
from pathlib import Path
import numpy as np
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path('/kaggle/working'); FIG=OUT/'figures'; RES=OUT/'results'
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device',device,'gpus',torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}; GATES={}
def gate(name, ok, detail=''):
    GATES[name]=bool(ok); print(('PASS' if ok else 'FAIL'), name, detail)
    if not ok: raise AssertionError(f'ACCEPTANCE FAILED: {name} {detail}')
def make_shape_image(kind, size=64):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=='red_circle':
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=='blue_square':
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=='green_triangle':
        top=cy-int(size*0.28); bot=cy+int(size*0.30)
        for y in range(max(0,top),min(size,bot)):
            half=int((y-top)/max(1,bot-top)*size*0.30)
            img[y, max(0,cx-half):min(size,cx+half+1)]=(0.15,0.75,0.25)
    else: raise ValueError(kind)
    return img
CLASSES=['red_circle','blue_square','green_triangle']
c2i={c:i for i,c in enumerate(CLASSES)}
CAPTIONS={c:'a '+c.replace('_',' ') for c in CLASSES}


## FS11


In [ ]:
# Conditional image generator: supervised MLP (stable) + optional noise residual demo
class ImgGen(nn.Module):
    def __init__(self,size=32,z=16,n_cls=3):
        super().__init__(); self.size=size
        self.emb=nn.Embedding(n_cls,32)
        self.net=nn.Sequential(nn.Linear(z+32,256),nn.ReLU(),nn.Linear(256,512),nn.ReLU(),nn.Linear(512,3*size*size),nn.Sigmoid())
    def forward(self,z,y):
        return self.net(torch.cat([z,self.emb(y)],-1)).view(z.size(0),3,self.size,self.size)

gen=ImgGen().to(device); opt=torch.optim.Adam(gen.parameters(),lr=2e-3)
hist11=[]
for epoch in range(1,50):
    gen.train(); losses=[]
    for _ in range(40):
        ys=[]; xs=[]
        for _b in range(64):
            k=random.choice(CLASSES); ys.append(c2i[k])
            img=make_shape_image(k,32); xs.append(img.transpose(2,0,1))
        y=torch.tensor(ys,device=device); x=torch.tensor(np.stack(xs),dtype=torch.float32,device=device)
        z=torch.randn(len(ys),16,device=device)*0.1  # small noise
        loss=F.mse_loss(gen(z,y),x)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(loss.item())
    hist11.append({'epoch':epoch,'mse':round(float(np.mean(losses)),6)})
    if epoch%10==0: print(hist11[-1])

@torch.no_grad()
def gen_img(kind):
    gen.eval(); z=torch.zeros(1,16,device=device); y=torch.tensor([c2i[kind]],device=device)
    return gen(z,y)[0].cpu().numpy().transpose(1,2,0)

dists={}; mses={}; fig,axes=plt.subplots(2,3,figsize=(8,5))
for j,k in enumerate(CLASSES):
    gt=make_shape_image(k,32); g=np.clip(gen_img(k),0,1)
    dists[k]=float(np.linalg.norm(g.mean((0,1))-gt.mean((0,1))))
    mses[k]=float(np.mean((g-gt)**2))
    axes[0,j].imshow(gt); axes[0,j].set_title('GT '+k,fontsize=8); axes[0,j].axis('off')
    axes[1,j].imshow(g); axes[1,j].set_title(f'mse={mses[k]:.4f}',fontsize=8); axes[1,j].axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs11_gen.png',dpi=120); plt.close()
print(mses,dists)
gate('FS11_mse', all(v<0.01 for v in mses.values()), mses)
gate('FS11_color', all(v<0.25 for v in dists.values()), dists)
(RES/'fs11.json').write_text(json.dumps({'stage':'FS11','method':'class-cond supervised image generator','history':hist11,'frame_mse':mses,'color_l2_to_gt_mean':dists,'vs_prev':'understand->generate pixels'},indent=2))
PROGRESS['FS11']='ok'


## FS12


In [ ]:
class VideoGen(nn.Module):
    def __init__(self,T=8,size=32,z=16,n_cls=3):
        super().__init__(); self.T=T; self.size=size
        self.emb=nn.Embedding(n_cls,32)
        self.fc=nn.Sequential(nn.Linear(z+32,512),nn.ReLU(),nn.Linear(512,T*3*size*size),nn.Sigmoid())
    def forward(self,z,y):
        return self.fc(torch.cat([z,self.emb(y)],-1)).view(z.size(0),self.T,3,self.size,self.size)

def make_video_gt(kind,T=8,size=32):
    frames=[]
    for t in range(T):
        img=np.ones((size,size,3),np.float32)*0.95
        yy,xx=np.mgrid[0:size,0:size]
        if kind=='red_circle':
            cx=int(size*(0.25+0.5*t/(T-1))); cy=size//2; r=size*0.16
            m=(yy-cy)**2+(xx-cx)**2<=r**2; img[m]=(0.9,0.15,0.12)
        elif kind=='blue_square':
            cy=int(size*(0.25+0.5*t/(T-1))); cx=size//2; s=int(size*0.16)
            m=(np.abs(yy-cy)<s)&(np.abs(xx-cx)<s); img[m]=(0.15,0.25,0.85)
        else:
            cy,cx=size//2,size//2; sc=0.12+0.08*(t/(T-1))
            top=int(cy-size*sc); bot=int(cy+size*sc*1.2)
            for y in range(max(0,top),min(size,bot)):
                half=int((y-top)/max(1,bot-top)*size*sc)
                img[y, max(0,cx-half):min(size,cx+half+1)]=(0.15,0.75,0.25)
        frames.append(img.transpose(2,0,1))
    return np.stack(frames)

vg=VideoGen().to(device); opt=torch.optim.Adam(vg.parameters(),lr=2e-3)
hist12=[]
for epoch in range(1,50):
    vg.train(); losses=[]
    for _ in range(40):
        ys=[]; vs=[]
        for _b in range(32):
            k=random.choice(CLASSES); ys.append(c2i[k]); vs.append(make_video_gt(k))
        y=torch.tensor(ys,device=device); v=torch.tensor(np.stack(vs),dtype=torch.float32,device=device)
        z=torch.randn(len(ys),16,device=device)*0.05
        loss=F.mse_loss(vg(z,y),v)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(loss.item())
    hist12.append({'epoch':epoch,'mse':round(float(np.mean(losses)),6)})
    if epoch%10==0: print(hist12[-1])

@torch.no_grad()
def gen_video(kind):
    vg.eval(); z=torch.zeros(1,16,device=device); y=torch.tensor([c2i[kind]],device=device)
    return vg(z,y)[0].cpu().numpy().transpose(0,2,3,1)

mses={}; fig,axes=plt.subplots(3,8,figsize=(12,4.5))
for i,k in enumerate(CLASSES):
    gt=make_video_gt(k).transpose(0,2,3,1); g=np.clip(gen_video(k),0,1)
    mses[k]=float(np.mean((gt-g)**2))
    for t in range(8):
        axes[i,t].imshow(g[t]); axes[i,t].axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs12_video_gen.png',dpi=120); plt.close()
print(mses)
gate('FS12_mse', all(v<0.01 for v in mses.values()), mses)
(RES/'fs12.json').write_text(json.dumps({'stage':'FS12','method':'cond video generator','history':hist12,'frame_mse':mses,'vs_prev':'image gen->video gen'},indent=2))
PROGRESS['FS12']='ok'


In [ ]:
(RES/'summary_fs11_fs12.json').write_text(json.dumps({'progress':PROGRESS,'gates':GATES},indent=2))
(OUT/'SUCCESS').write_text('ok\n'); (OUT/'ACCEPTANCE.json').write_text(json.dumps({'ok':all(GATES.values()),'gates':GATES},indent=2))
print('FS11-12 PASS',GATES)
